In [21]:
# Import packages

import kagglehub
import os
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
#import joblib
import lightgbm as lgb
#import mlflow
#import optuna


In [22]:
# Get project location
parent_dir = os.path.dirname(os.getcwd())

input_folder_dir = fr"{parent_dir}\input_files"
final_output_dir = fr"{parent_dir}\final_output"

# Create folders if they don't exist
if not os.path.exists(input_folder_dir):
    os.makedirs(input_folder_dir)

if not os.path.exists(final_output_dir):
    os.makedirs(final_output_dir)

    
# Download titanic competition dataset if not already downloaded

if not os.listdir(input_folder_dir):
    download_path = kagglehub.competition_download("titanic",output_dir=rf"{parent_dir}\input_files")

df = {}
df['train'] = pd.read_csv(os.path.join(input_folder_dir, "train.csv"))
df['test'] = pd.read_csv(os.path.join(input_folder_dir, "test.csv"))

In [23]:
# Feature engineering
for x in df:
    df[x]['est_age_flag'] = np.where(df[x]['Age'].notna() & (abs(df[x]['Age'] - 0.5) - np.floor(df[x]['Age'])) <0.1   , 1 , 0 ) # flag for if age is estimated in given data
    df[x]['imp_age_flag'] = np.where(df[x]['Age'].isna(), 1 , 0 ) # flag for if age will be imputed
    df[x]['Age'] = df[x]['Age'].fillna(df['train']['Age'].median()) # always impute age with training median
    df[x]['cabin_letter'] = df[x]['Cabin'].str[0].fillna("U") # Cabin letter (U if unknown)
    df[x]['Embarked'] = df[x]['Embarked'].fillna("U") # Impute Embarked (U if unknown)
    df[x]['is_mr'] = np.where(df[x]['Name'].str.contains("Mr.", regex=False) ,1,0) # Title is "Mr."
    df[x]['is_mrs'] = np.where(df[x]['Name'].str.contains("Mrs.", regex=False),1,0) # Title is "Mrs."
    df[x]['is_miss'] = np.where(df[x]['Name'].str.contains("Miss", regex=False),1,0) # Title is "Miss"
    df[x]['is_master'] = np.where(df[x]['Name'].str.contains("Master", regex=False),1,0) # Title is "Master"

# Remove unwanted columns
cols_to_drop = ["Survived", "PassengerId", "Name", "Ticket", "Cabin"]
X_train = df["train"].drop(columns=cols_to_drop).copy()
X_test = df["test"].drop(columns=cols_to_drop, errors="ignore").copy()
y_train = df["train"]["Survived"].copy()


# define categorical and numeric features
numeric_features = X_train.select_dtypes(exclude=['object', 'string']).columns
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns



In [24]:
print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")

is_cols = [x for x in X_train.columns if x[0:3] == "is_"]
X_train[is_cols].sum(axis = 0)



Numeric features: Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'est_age_flag',
       'imp_age_flag', 'is_mr', 'is_mrs', 'is_miss', 'is_master'],
      dtype='str')
Categorical features: Index(['Sex', 'Embarked', 'cabin_letter'], dtype='str')


is_mr        517
is_mrs       125
is_miss      182
is_master     40
dtype: int64

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough",
)

full_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            lgb.LGBMClassifier(verbosity=-1),
        ),
    ]
)

param_grid = {
    "classifier__n_estimators": [100,200],
    "classifier__learning_rate": [0.008, 0.002, 0.005],
    "classifier__num_leaves": [4, 7, 12, 15],
    "classifier__max_depth": [3, 4, 5],  
    "classifier__min_child_samples": [20, 30, 40],  
    "classifier__random_state": [1],
}

grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring={"accuracy": "accuracy", "roc_auc": "roc_auc"},
    refit="roc_auc",
    return_train_score=True,
    n_jobs=-1,
)

# Fit the new model
grid_search.fit(X_train, y_train)


results = grid_search.cv_results_
best_index = grid_search.best_index_

print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Train set - Mean CV accuracy: {results['mean_train_accuracy'][best_index]:.4f}")
print(f"Validation set - Mean CV accuracy: {results['mean_test_accuracy'][best_index]:.4f}")
print(f"Train set - Mean AUC ROC accuracy: {results['mean_train_roc_auc'][best_index]:.4f}")
print(f"Validation set - Mean AUC ROC accuracy: {results['mean_test_roc_auc'][best_index]:.4f}")

# Generate LightGBM Predictions
final_preds = grid_search.predict(X_test)
submission = pd.DataFrame(
    {"PassengerId": df["test"]["PassengerId"], "Survived": final_preds}
)
print(submission["Survived"].value_counts())
submission.to_csv(rf"{final_output_dir}/final_submission.csv", index=False)

Best Parameters Found: {'classifier__learning_rate': 0.03, 'classifier__max_depth': 5, 'classifier__min_child_samples': 40, 'classifier__n_estimators': 90, 'classifier__num_leaves': 15, 'classifier__random_state': 1}
Train set - Mean CV accuracy: 0.8493
Validation set - Mean CV accuracy: 0.8227
Train set - Mean AUC ROC accuracy: 0.9120
Validation set - Mean AUC ROC accuracy: 0.8714
Survived
0    274
1    144
Name: count, dtype: int64


D:\Python\venv\main_venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
